# LifeMate trên Colab

Chuẩn bị một lần:
1. **Runtime → Change runtime type → GPU A100** (L4 cũng chạy được, chậm hơn). T4 có thể không chạy được vLLM bản mới.
2. Upload `train+public_test.zip` (có sẵn trong folder project trên Mac) lên Google Drive tại `MyDrive/zalo_lifemate/train+public_test.zip`.
3. Tạo GitHub token (fine-grained, quyền *Contents: Read* cho repo `FirstKhoi/zalo_lifemate`), thêm vào **Colab Secrets** (biểu tượng chìa khoá bên trái) với tên `GH_TOKEN` và bật *Notebook access*.

Nếu sau cell cài đặt mà một dòng `!...` báo `SyntaxError`: **Thời gian chạy → Khởi động lại phiên**, chạy lại cell 1 rồi đi tiếp (không cần cài lại).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + data

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
assert os.path.isfile(f"{DRIVE}/train+public_test.zip"), f"chưa có {DRIVE}/train+public_test.zip trên Drive"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!ls train+public_test

In [ ]:
# torchao cài sẵn trên Colab quá cũ, peft mới từ chối import khi có nó; project không dùng torchao
!pip install -q vllm peft && pip uninstall -y -q torchao

In [ ]:
!python test_score.py && python test_agent.py && python test_asr.py

## 2. Fine-tune LoRA trên splits/train (1223 mẫu, 2 epoch)
Bỏ qua cell train nếu đã có adapter trên Drive (`MyDrive/zalo_lifemate/lora`). Cell sau tự merge adapter thành `checkpoints/merged` cho vLLM khi cần.

In [ ]:
!python train_sft.py --out "{DRIVE}/lora" --merge-to checkpoints/merged 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"

In [ ]:
# checkpoints/merged nằm trên ổ tạm của Colab: phiên mới thì merge lại từ adapter trên Drive (~2 phút)
if not os.path.isdir("checkpoints/merged"):
    !python -c "import torch; from peft import AutoPeftModelForCausalLM as M; from transformers import AutoTokenizer as T; m = M.from_pretrained('{DRIVE}/lora', dtype=torch.bfloat16).merge_and_unload(); m.save_pretrained('checkpoints/merged'); T.from_pretrained('Qwen/Qwen3-4B-Instruct-2507').save_pretrained('checkpoints/merged')"
!ls checkpoints/merged

## 3. Đo dev bằng transcript chuẩn (audio_text)
Điểm trần của agent khi nghe đúng 100%. Đã chạy rồi thì bỏ qua (lần trước: 54.27 / 47.08).

In [ ]:
for split in ["dev_seen", "dev_unseen"]:
    !python agent.py --model checkpoints/merged --ids splits/{split}.txt --out runs/sft_{split}.jsonl 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"
    !python score.py runs/sft_{split}.jsonl

## 4. ASR: so sánh 2 model trên dev (WER = tỉ lệ từ nghe sai, càng thấp càng tốt)
Ngân sách tham số: Qwen3-4B (4.0B) + PhoWhisper-large (1.55B) = 5.6B, hoặc + whisper-large-v3-turbo (0.8B) = 4.8B, đều dưới 6B.

In [ ]:
ASR = {"pho": "vinai/PhoWhisper-large", "turbo": "openai/whisper-large-v3-turbo"}
for tag, model in ASR.items():
    for split in ["dev_seen", "dev_unseen"]:
        !python asr.py --ids splits/{split}.txt --model {model} --out runs/asr_{tag}_{split}.json 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"

## 5. Đo dev với transcript từ ASR
Đặt `BEST` là model có WER thấp hơn ở bước 4. Điểm tụt so với bước 3 = điểm mất do nghe sai.

In [ ]:
BEST = "pho"
for split in ["dev_seen", "dev_unseen"]:
    !python agent.py --model checkpoints/merged --ids splits/{split}.txt --asr runs/asr_{BEST}_{split}.json --out runs/asr_{BEST}_sft_{split}.jsonl 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"
    !python score.py runs/asr_{BEST}_sft_{split}.jsonl

## 6. Bài nộp public
Tạo `runs/public_test.jsonl`, kiểm tra schema bằng validator của BTC, chép lên Drive. Tải file đó về rồi nộp ở mục Submission trên portal.

In [ ]:
!python asr.py --split public_test --model {ASR[BEST]} --out runs/asr_{BEST}_public_test.json 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"
!python agent.py --split public_test --model checkpoints/merged --asr runs/asr_{BEST}_public_test.json --out runs/public_test.jsonl 2>&1 | grep --line-buffered -E "^\[|conversations|epoch|merged|clips|WER|Error|Traceback"
!wc -l runs/public_test.jsonl
!cd train+public_test/public_test && python -m zaic_tools --split public_test --root .. --validate ../../runs/public_test.jsonl

In [ ]:
!mkdir -p "{DRIVE}/runs" && cp runs/*.json runs/*.jsonl "{DRIVE}/runs/" && ls "{DRIVE}/runs"